# Rizzler V26 — Audited Gemini + large-context qualification (Kaggle)

This notebook tests the **current V26 feature branch**, not V24. It pins the exact Git commit during the run, verifies its SHA-256 release inventory, runs expanded V26 offline regressions, executes real paid Gemini multi-artifact Work jobs, and separately benchmarks out-of-core CSV data. **Nothing in this notebook claims production certification.** V26 production release requires 18 evidence gates, independent operational signoff, human quality review, and measured provider/service SLOs.

**Kaggle setup:** Internet ON; Secrets → `GEMINI_API_KEY` enabled for this notebook; CPU sufficient, T4 optional. Real Gemini use incurs API billing. Never print your credential or upload private sources in the evidence ZIP.

In [ ]:
import os, sys, subprocess, pathlib, hashlib, json, secrets, shutil, time, csv
from datetime import datetime, timezone
from IPython.display import display, FileLink

ROOT=pathlib.Path("/kaggle/working")
BRANCH="feature/v26-final-integration-gates"
URL="https://github.com/SAUNAK359/Voice-AI.git"
REPO=ROOT/"rizzler-v26-audit"
RUN=ROOT/("V26_AUDIT_"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+secrets.token_hex(3))
RUN.mkdir(parents=True,exist_ok=False)
if not (REPO/".git").exists():
    subprocess.run(["git","clone","--depth","1","--single-branch","--branch",BRANCH,URL,str(REPO)],check=True)
else:
    subprocess.run(["git","-C",str(REPO),"fetch","--depth","1","origin",BRANCH],check=True)
    subprocess.run(["git","-C",str(REPO),"checkout","--detach","FETCH_HEAD"],check=True)
SOURCE_SHA=subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()
assert len(SOURCE_SHA)==40 and all(c in "0123456789abcdef" for c in SOURCE_SHA)
subprocess.run(["git","-C",str(REPO),"checkout","--detach",SOURCE_SHA],check=True)
assert subprocess.check_output(["git","-C",str(REPO),"status","--porcelain"],text=True).strip()==""
(RUN/"AUDITED_SOURCE.json").write_text(json.dumps({"source_sha":SOURCE_SHA,"branch":BRANCH,"checked_at":datetime.now(timezone.utc).isoformat()},indent=2))
print("IMMUTABLE EXECUTION SHA:",SOURCE_SHA)
print("Evidence directory:",RUN)


In [ ]:
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(REPO)+"[work,test]","pymupdf>=1.25,<2"],check=True)
components=["libreoffice-writer","libreoffice-impress","libreoffice-calc"]
def installed(name):
    p=subprocess.run(["dpkg-query","-W","-f=${Status}",name],capture_output=True,text=True)
    return p.returncode==0 and "install ok installed" in p.stdout
missing=[x for x in components if not installed(x)]
if missing:
    subprocess.run(["apt-get","update","-qq"],check=True)
    subprocess.run(["apt-get","install","-y","-qq",*missing],check=True)
assert all(installed(x) for x in components),"Missing strict Office render dependencies"
manifest=subprocess.run([sys.executable,str(REPO/"scripts"/"verify_release.py")],cwd=REPO,check=False)
assert manifest.returncode==0,"Release manifest mismatch: do not run unverified source"
print("Environment and exact source inventory qualified.")


In [ ]:
from kaggle_secrets import UserSecretsClient
key=UserSecretsClient().get_secret("GEMINI_API_KEY")
if not key:
    raise RuntimeError("Enable Kaggle Secret GEMINI_API_KEY for this notebook")
env=os.environ.copy()
env["GEMINI_API_KEY"]=key
del key
MODEL_OVERRIDE=""  # Leave blank to use repo Settings; set only a valid available paid model ID.
print("Paid credential loaded; value not printed")


In [ ]:
command=[sys.executable,str(REPO/"scripts"/"qualify_v26_live.py"),
         "--repo",str(REPO),"--output",str(RUN),"--expected-sha",SOURCE_SHA]
if MODEL_OVERRIDE.strip():
    command+=["--model",MODEL_OVERRIDE.strip()]
result=subprocess.run(command,cwd=REPO,env=env,check=False)
verdict_file=RUN/"V26_QUALIFICATION.json"
if not verdict_file.exists():
    raise RuntimeError("Runner failed before writing verdict; check output")
verdict=json.loads(verdict_file.read_text())
assert verdict.get("source_sha")==SOURCE_SHA,"Wrong source in report"
print("OFFLINE:",verdict.get("offline_passed"),"LEGACY REAL GEMINI:",verdict.get("live_passed"),
      "V26 STREAMING REAL GEMINI:",verdict.get("streamed_paid_passed"),
      "RUNNER STATUS:",verdict.get("release_status"))
print("Process return code:",result.returncode)
if not verdict.get("passed"):
    print("Live qualification did NOT pass. Examine sanitized JSON and logs in the evidence ZIP.")


In [ ]:
# This is an independent, SOURCE-HASHED actual-file benchmark, not fake 200 MB proof.
# 12 MB smoke is ON. For a genuine >=190 MB benchmark, explicitly set RUN_200MB_BENCHMARK=True.
RUN_200MB_BENCHMARK=False
target_bytes=190_000_000 if RUN_200MB_BENCHMARK else 12_000_000
source=RUN/"private_benchmark_source.csv"  # Excluded from evidence bundle.
payload="s"*1150
with source.open("w",encoding="utf-8",newline="") as handle:
    writer=csv.writer(handle)
    writer.writerow(["region","imports","exports","description"])
    n=0
    while handle.tell()<target_bytes:
        writer.writerow(["North" if n%2 else "South",100+n%17,95+n%11,payload])
        n+=1
print("Prepared local fixture size:",source.stat().st_size,"rows:",n)
perf=RUN/"STREAMING_BENCHMARK.json"
command=[sys.executable,str(REPO/"scripts"/"qualify_v26_large_data.py"),
         "--source",str(source),"--output",str(perf),
         "--min-bytes",str(target_bytes),"--max-seconds","900","--max-rss-mb","2048"]
p=subprocess.run(command,cwd=REPO,env=env,check=False)
if perf.exists():
    metrics=json.loads(perf.read_text())
    print("BENCHMARK:",metrics.get("status"),"ROWS:",metrics.get("source_rows"),
          "SECONDS:",metrics.get("elapsed_seconds"),"RSS MB:",metrics.get("peak_rss_mb_linux"))
else:
    print("Benchmark stopped before producing machine-readable evidence.")
if p.returncode:
    print("Benchmark NOT QUALIFIED; open evidence JSON for failure category.")
# Do not include even synthetic 190 MB raw fixture in the evidence bundle.
source.unlink(missing_ok=True)


In [ ]:
# Audit all 18 gates. Missing proofs MUST remain NOT_RUN.
evaluation=RUN/"V26_RELEASE_DECISION.json"
decision=subprocess.run([
    sys.executable,str(REPO/"scripts"/"evaluate_v26_release.py"),
    "--evidence",str(RUN),
    "--scope","research","--scope","editable-artifacts",
    "--scope","streamed-csv-analytics",
    "--output",str(evaluation),
    "--source-sha",SOURCE_SHA],cwd=REPO,env=env,check=False)
if evaluation.exists():
    release=json.loads(evaluation.read_text())
    print("ACTUAL RELEASE STATUS:",release.get("status"))
    print("Not-run gates:",release.get("not_run"))
    print("Failed gates:",release.get("failed"))
print("Exit code 2 is expected when production qualification has NOT been established:",decision.returncode)


In [ ]:
# Create one clean evidence bundle with content hashes; explicitly exclude raw sources and secrets.
import zipfile
out=ROOT/("Rizzler_V26_Audit_"+SOURCE_SHA[:12]+"_"+RUN.name+".zip")
excluded_dirs={"private_workstore","private_slots","private_benchmark_source.csv","live_fixture"}
excluded_ext={".db",".sqlite3",".key",".pem",".env",".p12"}
with zipfile.ZipFile(out,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RUN.rglob("*")):
        if not f.is_file():
            continue
        rel=f.relative_to(RUN)
        if any(part.startswith(".") or part in excluded_dirs for part in rel.parts):
            continue
        if f.suffix.lower() in excluded_ext or f.stat().st_size>20_000_000:
            continue
        z.write(f,arcname=rel.as_posix())
print("Audited SHA:",SOURCE_SHA,"Evidence ZIP:",out.name)
display(FileLink(str(out)))
print("This bundle is not a release certificate. Production gates require independent evidence and operator approval.")
